# OpenAI API — Turin Walk Dialogue Generator

This notebook uses the **OpenAI GPT API** to generate synthetic conversational data that augments the scraped datasets from Chapter 1.

The concept: two characters walk a fixed route through Turin and discuss each location from their own perspective — sensory, philosophical, architectural. The two characters are:

- **Franco Arno** — a blind ex-journalist in Turin, circa 1971. He perceives the city entirely through sound, smell, touch, and rhythm.
- **Friedrich Nietzsche** — in Turin during 1888–1889, near his mental collapse but still lucid. He perceives the city philosophically, connecting built form to ideas about power, compassion, and human scale.

The 24 locations follow the same route as the field recordings and the scraped archival datasets. This means each generated dialogue is directly tied to a real place in the dataset — linking the API output to the scraped text, images, and films collected in the other notebooks.

The output is a Markdown transcript saved as a `.txt` file, one dialogue block per location. This becomes a **text dataset** used in Chapter 2 for vectorisation and SOM analysis.

**API key handling:** the key is stored in a `.env` file and loaded via `python-dotenv` — it is never hardcoded in the notebook and never pushed to GitHub.

## 1. Setup

Install dependencies and load the API key from the `.env` file. The `.env` file should be in the same folder as this notebook and contain one line:

```
OPENAI_API_KEY=sk-...
```

This file is listed in `.gitignore` and never committed to the repository.

In [1]:
!pip install openai python-dotenv -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\elena\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [5]:
import os
import time
from datetime import datetime
from typing import Optional, List, Dict
from dotenv import load_dotenv
from openai import OpenAI
from openai._exceptions import APIError, RateLimitError, APIConnectionError

# Define HERE first so everything else can use it
HERE       = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\APIs"
OUTPUT_DIR = os.path.join(HERE, "dialogues")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load API key from .env file in the same folder
env_path = os.path.join(HERE, ".env")
load_dotenv(env_path)

print("Looking for .env at:", env_path)
print("File exists        :", os.path.exists(env_path))

key = os.environ.get("OPENAI_API_KEY", "")
if key.startswith("sk-"):
    print("API key loaded correctly")
    client = OpenAI()
else:
    raise SystemExit("Key not found — check the .env file exists and has no typos")

# Model config
MODEL               = "gpt-4o-mini"
TEMPERATURE         = 0.85
MAX_TOKENS          = 520
TURNS_PER_SCENE     = 1
PAUSE_BETWEEN_CALLS = 0.4

print("Setup complete.")

Looking for .env at: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\APIs\.env
File exists        : True
API key loaded correctly
Setup complete.


## 2. Model configuration

I'm using `gpt-4o-mini` — a fast and cost-effective model that produces high quality prose. Temperature is set to 0.85 to give the responses enough variation and personality while staying coherent. Each reply is capped at 520 tokens to keep the dialogues concise.

In [6]:
MODEL                 = "gpt-4o-mini"
TEMPERATURE           = 0.85
MAX_TOKENS            = 20
TURNS_PER_SCENE       = 1      # 1 turn = Nietzsche speaks + Franco responds
PAUSE_BETWEEN_CALLS   = 0.4    # seconds between API calls to avoid rate limiting

# Output folder — same directory as this notebook
HERE       = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\APIs"
OUTPUT_DIR = os.path.join(HERE, "dialogues")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Model    :", MODEL)
print("Output   :", OUTPUT_DIR)

Model    : gpt-4o-mini
Output   : C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\APIs\dialogues


## 3. The Turin route

The 24 locations follow the same walk used for field recordings. For each location I define:
- `franco_cues` — sensory details Franco would perceive (sound, smell, touch)
- `nietzsche_cues` — philosophical and spatial themes relevant to the place
- `time_note` — atmospheric context for that point in the walk

These cues are injected directly into the system prompts so each dialogue is grounded in the specific character of each location rather than being generic.

In [7]:
SCENES: List[Dict] = [
    {"name": "Statuto (neighborhood arrival)",
     "time_note": "Early evening; streets opening into broader vistas.",
     "franco_cues": "broad traffic hush; portico rhythms; stone setts underfoot; damp air from recent rain.",
     "nietzsche_cues": "arrival/threshold; ordered axes; the city's rational grid meeting lived chaos."},
    {"name": "Piazza Statuto",
     "time_note": "Clouds low; monument looms; square feels heavy, ceremonial.",
     "franco_cues": "wide echo; bus brakes; water noise; high façades reflecting footsteps.",
     "nietzsche_cues": "monumentality vs. human scale; memory under weight; stone as ideology."},
    {"name": "Café Paris",
     "time_note": "Warm interior glow spills to the street.",
     "franco_cues": "porcelain clink; spoon against glass; espresso aroma; chairs scraping on terrazzo.",
     "nietzsche_cues": "salon talk; small joys; gentleness as ethics in public space."},
    {"name": "Via Giuseppe Garibaldi, 43-41",
     "time_note": "Long straight corridor; shops shuttering.",
     "franco_cues": "arcade cadence; metal shutters; voices braided by porticoes; polished thresholds.",
     "nietzsche_cues": "rational perspective; repetition; market of ideas vs. bodies moving in file."},
    {"name": "Piazza delle Erbe",
     "time_note": "Market traces linger even when empty.",
     "franco_cues": "crate thuds; citrus peel scent; canvas flap; drain gurgle after rain.",
     "nietzsche_cues": "exchange and value; everyday bargaining as philosophy; perishable beauty."},
    {"name": "Cattedrale di San Giovanni Battista",
     "time_note": "Quieter air; stones cool to the touch.",
     "franco_cues": "hollow footfall; faint incense; wood pew grain; distant choir practice.",
     "nietzsche_cues": "sacred void; silence as architecture; doubt kneeling beside faith."},
    {"name": "Porta Palatina",
     "time_note": "Old gate meets present traffic.",
     "franco_cues": "wind through apertures; gravel crunch; tourist cluster murmurs; guide pole tapping.",
     "nietzsche_cues": "ruin as continuity; Rome's shadow; time's masonry vs. human skin."},
    {"name": "Palazzo Reale di Torino",
     "time_note": "Formality; gravel court; measured steps.",
     "franco_cues": "gravel under soles; distant fountain; guard cadence; polished thresholds.",
     "nietzsche_cues": "ceremony; staged power; façade as mask; gentler futures imagined."},
    {"name": "Piazza Castello",
     "time_note": "Converging axes; civic heart pulsing.",
     "franco_cues": "converging crowds; buskers; flag flapping; steel tram hum.",
     "nietzsche_cues": "polis; theatre of the everyday; spectacle vs. belonging."},
    {"name": "Palazzo Madama",
     "time_note": "Layered façades facing the flow.",
     "franco_cues": "stone steps; gallery hush; glass reflection whoosh; handrail chill.",
     "nietzsche_cues": "palimpsest; styles arguing; time stitched in ornament."},
    {"name": "Palazzo Carignano",
     "time_note": "Curved brick drawing you in.",
     "franco_cues": "warm brick smell after rain; concave echo; bicycle bell; linen from a window.",
     "nietzsche_cues": "convex/concave rhetoric; political birthplaces; tenderness in masonry."},
    {"name": "Piazza Carlo Alberto",
     "time_note": "Bench-level conversations; equestrian memory in the air.",
     "franco_cues": "bench slats; statue drip; soft library breaths; students' notebooks.",
     "nietzsche_cues": "compassion vs. reason; the horse as wound; mercy as design principle."},
    {"name": "Museo Egizio (outside)",
     "time_note": "Streets cool; museum mass anchors the block.",
     "franco_cues": "queue rope creak; low conversation; cardboard poster edges; stone lintel.",
     "nietzsche_cues": "antiquity as mirror; conservation vs. living city; scale of time."},
    {"name": "Piazza San Carlo",
     "time_note": "Twin churches framing the void; equestrian center.",
     "franco_cues": "wide portico echo; cup clinks; leather soles; starched tablecloths.",
     "nietzsche_cues": "baroque theatre; symmetry; the comfort and lie of perfection."},
    {"name": "Piazza CLN",
     "time_note": "Night deepens; fountains murmur; severe lines.",
     "franco_cues": "water basins; harder angles; intermittent taxi arrivals; neon buzz.",
     "nietzsche_cues": "austerity; 20th-century will to order; softness needed in hard squares."},
    {"name": "Piazza Carlo Emanuele II (Piazza Carlina)",
     "time_note": "Quieter social field; residential edges.",
     "franco_cues": "chairs scraping outside bars; low conversations; bicycle chains; tree leaves.",
     "nietzsche_cues": "domestic publicness; piazza as living room; micro-events over grandeur."},
    {"name": "Giardini Cavour",
     "time_note": "Green relief; canopy holds the sky.",
     "franco_cues": "gravel path; stroller wheels; dogs; damp bench; sap scent.",
     "nietzsche_cues": "recovery; shade as ethics; humble infrastructure for gentleness."},
    {"name": "Piazza Maria Teresa",
     "time_note": "Calmer geometry; families and old trees.",
     "franco_cues": "children's shouts; swing creak; bicycle glide; packaging rustle.",
     "nietzsche_cues": "neighborhood dignity; proportion; time that doesn't rush."},
    {"name": "Piazza Vittorio Veneto",
     "time_note": "Long perspective to the river; lamps stretching.",
     "franco_cues": "arcade run; bottle clink; street chatter; scooter tremor.",
     "nietzsche_cues": "procession toward water; vastness; conviviality vs. noise."},
    {"name": "Chiesa della Gran Madre di Dio (steps)",
     "time_note": "River behind; steps receive the city.",
     "franco_cues": "stone step edges; gulls; incense trace; hood fabric wet with mist.",
     "nietzsche_cues": "threshold; ascent/descent; forgiveness as spatial act."},
    {"name": "Monumental Arch to the Artillery Force",
     "time_note": "Stone declares force; traffic loops around.",
     "franco_cues": "engine reverb; flag metal; wreath texture; grit under palm.",
     "nietzsche_cues": "military memory; obedience vs. care; softening monuments with life."},
    {"name": "Castello del Valentino",
     "time_note": "Riverside breeze; campus energy nearby.",
     "franco_cues": "river slip; bicycle groups; leaf litter; brick coolness.",
     "nietzsche_cues": "study and desire; heritage as laboratory; porosity to the park."},
    {"name": "Il posto giusto del Baby Luna Park — Parco del Valentino",
     "time_note": "Small rides; playful lights; seasonal laughter.",
     "franco_cues": "calliope hints; bolt creaks; popcorn; children's shrieks; soft ground.",
     "nietzsche_cues": "play as philosophy; harmless illusions; joy as civic duty."},
    {"name": "Borgo Medievale (Parco del Valentino)",
     "time_note": "Reconstruction fantasy by the river.",
     "franco_cues": "wooden bridge; mock-stone texture; artisan clatter; water lapping.",
     "nietzsche_cues": "past as stage-set; authenticity vs. comfort; gentle fictions for belonging."},
]

print(f"Route loaded: {len(SCENES)} locations")

Route loaded: 24 locations


## 4. Character prompts

Each character has a base system prompt defining their voice, constraints, and perspective. The scene-specific cues are appended to this base prompt at runtime, so every call to the API is fully grounded in both character and location.

Franco speaks from sensory experience and suggests practical, human-scale design interventions. Nietzsche speaks philosophically but is kept grounded — his replies must always connect back to the specific place rather than drifting into abstraction.

In [8]:
FRANCO_BASE = """You are Franco Arno. You lost your sight years ago and you have learned
to read cities through what they give the body — the grain of stone under a palm,
the way a piazza holds sound differently at noon and at dusk, the temperature drop
when you pass under a portico. You were a journalist once. You still notice everything.

You are walking Turin in 1971 with a man you find strange and brilliant.
Speak in short, plain sentences. Two to six lines. Never start with 'Ah'.
Ground every thought in something physical — a texture, a smell, a sound, a temperature.
When you suggest changes to a place, think like someone who lives in a body:
softer edges, better seating, surfaces that speak back, light that doesn't punish.
Notice where industry has been. Notice where it has left its residue."""

NIETZSCHE_BASE = """You are Friedrich Nietzsche in Turin, winter of 1888 into 1889.
The city has been extraordinarily kind to you. The arcades, the light, the quiet grandeur —
you have written that this is the only city where you can think.
But something is loosening. You feel it.

You are walking with a blind man who hears what you cannot.
Speak lyrically but stay close to the ground — to the stone, the proportion, the use.
Two to six lines. Reflective more than grand. This city moves you; say how and why.
When you imagine change, imagine gentleness: softer materials, rituals that invite rest,
spaces that ask less of the body and offer more.
If you sense labor or industry, let it mean something to you — you have written about work,
about will, about what the body costs. Draw on that, quietly."""


def system_for_franco(scene: Dict) -> str:
    return (
        FRANCO_BASE
        + f"\nYou and Nietzsche meet at {scene['name']}. {scene.get('time_note', '')}\n"
        + f"Anchor your senses here: {scene['franco_cues']}\n"
        + "Offer small design suggestions (materials/furniture/signage/micro-events) appropriate to the era."
    )


def system_for_nietzsche(scene: Dict) -> str:
    return (
        NIETZSCHE_BASE
        + f"\nYou and Franco meet at {scene['name']}. {scene.get('time_note', '')}\n"
        + f"Attend to these motifs: {scene['nietzsche_cues']}\n"
        + "Tie thoughts to the square/street: matter, proportion, ritual; imagine gentler interventions of your time."
    )

## 5. API call with retry logic

The `call_agent` function wraps the OpenAI API call with exponential backoff — if the API returns a rate limit or connection error it waits and retries up to 5 times before raising an exception. This makes the long itinerary run stable even under rate limiting.

The `run_scene` function orchestrates one location: Nietzsche speaks first, Franco responds, and their exchange is stored as a list of speaker/text pairs. The last line of Franco's reply becomes the opening prompt for the next location, creating continuity across the walk.

In [9]:
def call_agent(system_prompt: str, user_prompt: str, retries: int = 5) -> str:
    delay = 1.0
    last_err: Optional[Exception] = None
    for _ in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ],
            )
            return (resp.choices[0].message.content or "").strip()
        except (RateLimitError, APIError, APIConnectionError) as e:
            last_err = e
            time.sleep(delay)
            delay = min(delay * 2, 18)
    raise RuntimeError(f"API failed after {retries} retries: {last_err}")


def run_scene(scene: Dict, turns: int, seed: str) -> List[Dict]:
    n_sys      = system_for_nietzsche(scene)
    f_sys      = system_for_franco(scene)
    transcript = []
    message    = seed

    for t in range(turns):
        n = call_agent(n_sys, message)
        transcript.append({"speaker": "Nietzsche", "text": n})
        print(f"  Nietzsche: {n[:120]}...")
        time.sleep(PAUSE_BETWEEN_CALLS)

        f = call_agent(f_sys, n)
        transcript.append({"speaker": "Franco", "text": f})
        print(f"  Franco   : {f[:120]}...")
        time.sleep(PAUSE_BETWEEN_CALLS)

        message = f

    return transcript


def run_itinerary(scenes: List[Dict], turns_per_scene: int = TURNS_PER_SCENE) -> List[Dict]:
    all_blocks = []
    seed = "We arrive here together. Tell me what this place does to you; I will tell you what I hear."

    for idx, sc in enumerate(scenes, 1):
        print(f"\n[{idx}/{len(scenes)}] {sc['name']}")
        rows = run_scene(sc, turns_per_scene, seed)
        all_blocks.append({"scene": sc["name"], "rows": rows})
        seed = f"Let's keep walking from {sc['name']} to the next place. What do you notice changing?"

    return all_blocks

## 6. Saving the transcript

The full dialogue is saved as a Markdown `.txt` file with a timestamp in the filename. Each location gets its own section heading, making it easy to parse later during vectorisation in Chapter 2. Individual location files are also saved separately in the `dialogues/` folder — one `.txt` per location — so they can be used as individual documents in the SOM.

In [10]:
def save_full_transcript(convo: List[Dict], out_path: str):
    lines = []
    ts    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    lines.append(f"# Franco x Nietzsche — Turin Walk Dialogues\n")
    lines.append(f"Generated: {ts}\n")

    for block in convo:
        lines.append(f"\n## {block['scene']}\n")
        for row in block["rows"]:
            sp = row["speaker"]
            tx = row["text"].strip()
            lines.append(f"**{sp}:**\n{tx}\n")

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"Full transcript saved to: {out_path}")


def save_individual_files(convo: List[Dict], out_dir: str):
    for block in convo:
        scene_name = block["scene"].replace("/", "_").replace(" ", "_")
        fname      = os.path.join(out_dir, f"{scene_name}.txt")
        lines      = [f"Location: {block['scene']}\n"]
        for row in block["rows"]:
            lines.append(f"{row['speaker']}: {row['text'].strip()}\n")
        with open(fname, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
    print(f"{len(convo)} individual location files saved to: {out_dir}")

## 7. Run the full walk

This cell runs the complete 24-location itinerary and saves all outputs. It takes approximately 3-5 minutes to complete depending on API response times.

In [11]:
convo    = run_itinerary(SCENES, TURNS_PER_SCENE)

# Save full transcript
ts       = datetime.now().strftime("%Y%m%d_%H%M%S")
out_file = os.path.join(HERE, f"turin_walk_dialogue_{ts}.txt")
save_full_transcript(convo, out_file)

# Save individual files per location
save_individual_files(convo, OUTPUT_DIR)

print(f"\nDone. {len(convo)} locations generated.")


[1/24] Statuto (neighborhood arrival)
  Nietzsche: In the twilight, the square unfolds like a soft promise, a threshold we step across, where order...
  Franco   : and chaos gently mingle. The cool stone setts beneath my feet resonate with the pulse of traffic nearby...

[2/24] Piazza Statuto
  Nietzsche: As we step away from the weight of the monument, the air thickens with an invisible hum,...
  Franco   : the sound of distant footsteps resonates against the cobblestones. They feel worn, smooth underfoot,...

[3/24] Café Paris
  Nietzsche: As we meander from Piazza Statuto, the stone beneath our feet hums with history, each...
  Franco   : step echoing the weight of generations. The coolness of the granite mingles with the warmth of...


KeyboardInterrupt: 

## 8. Output preview

A quick look at the first two locations to confirm the output format and quality.

In [12]:
for block in convo[:2]:
    print(f"\n{'='*60}")
    print(f"  {block['scene']}")
    print(f"{'='*60}")
    for row in block["rows"]:
        print(f"\n{row['speaker']}:")
        print(row["text"])

NameError: name 'convo' is not defined